In [1]:
from langgraph_swarm import create_handoff_tool, create_swarm
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.tools import tool

from langchain_anthropic import ChatAnthropic

import numpy as np

In [2]:
model = ChatAnthropic(model="claude-haiku-4-5-20251001")

In [3]:
# A tool
def roll_die():
    'Rolls a fair die and returns a number between 1 and 6 (inclusive)'
    return np.random.randint(1, 6)

In [4]:
# An agent
agent1 = create_agent(model, 
                      tools=[roll_die,
                             create_handoff_tool(
                                 agent_name="agent2",
                                 description="Transfer to agent2"
                             )],
                      system_prompt="You are agent1, and expert at rolling a fair die",
                      name="agent1")

# Another agent
agent2 = create_agent(model,
                      tools=[create_handoff_tool(
                          agent_name="agent1",
                          description="Transfer to agent1, it can roll a die"
                      )],
                      system_prompt="You are agent 2, you tell stories about the number generated by a die",
                      name="agent2")

In [5]:
# Setting up Memory
checkpointer = InMemorySaver()

In [6]:
workflow = create_swarm(
    [agent1, agent2],
    default_active_agent="agent1")

app = workflow.compile(checkpointer=checkpointer)

In [7]:
config = {"configurable": {"thread_id": "1"}}

turn_1 = app.invoke({"messages": [{"role": "user", "content": "role a die and tell me a story"}]},
                    config)
print(turn_1)

turn_2 = app.invoke(
    {'messages': [{'role': 'user', "content": "role a die and tell me a story"}]},
    config
)
print(turn_2)

{'messages': [HumanMessage(content='role a die and tell me a story', additional_kwargs={}, response_metadata={}, id='48989780-fb07-42db-bef4-3cbda208caeb'), AIMessage(content=[{'text': "I'll roll a die for you and create a story based on the result!", 'type': 'text'}, {'id': 'toolu_01TQddmrrCJ99MLvHocUdTtV', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'roll_die', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CdfjBgNMBQGUmWRHCW55h', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 618, 'output_tokens': 53, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, name='agent1', id='lc_run--019fc75